# CXR Disease Detection — Training Pipeline

Restart kernel and **Run All Cells** to train the three models in order:

1. **YOLOv8** — 10 epochs
2. **RT-DETR** — 10 epochs
3. **Faster R-CNN** — 10 epochs

Resume logic:
- If `last.pt` is resumable (has epoch+optimizer) → continue from where it stopped
- Else if `best.pt` already exists → training is complete, skip
- Else → train from scratch

In [2]:
%pip install -q ultralytics matplotlib


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import glob
import random
import shutil
import yaml
from pathlib import Path

from tqdm import tqdm
import pandas as pd
import numpy as np
import cv2

from IPython.display import display

import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader

import ultralytics
from ultralytics import YOLO, RTDETR

from sklearn.model_selection import train_test_split

print(f"PyTorch     : {torch.__version__}")
print(f"Ultralytics : {ultralytics.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")


PyTorch     : 2.8.0+cpu
Ultralytics : 8.4.52
CUDA        : False


In [4]:
# ----- Paths -----
PROJECT_ROOT = "Data"

DATA_ROOT = os.path.join(PROJECT_ROOT, "vindr_data")
CSV_PATH = os.path.join(DATA_ROOT, "train.csv")
META_PATH = os.path.join(DATA_ROOT, "train_meta.csv")
IMG_SOURCE_DIR = os.path.join(DATA_ROOT, "train")

YOLO_DATASET_DIR = os.path.join(PROJECT_ROOT, "yolo_dataset")
YAML_PATH = os.path.join(YOLO_DATASET_DIR, "data.yaml")

# Absolute paths for Ultralytics `project=` to avoid the runs/detect/ prefix bug
YOLO_RUN_DIR = os.path.abspath(os.path.join(PROJECT_ROOT, "YOLO_Runs"))
RTDETR_RUN_DIR = os.path.abspath(os.path.join(PROJECT_ROOT, "Transformer_Runs"))
FASTER_RCNN_SAVE_DIR = os.path.abspath(os.path.join(PROJECT_ROOT, "FasterRCNN_Runs"))
PRETRAINED_DIR = os.path.abspath(os.path.join(PROJECT_ROOT, "pretrained"))

YOLO_RUN_NAME = "yolov8n_cxr_test"
RTDETR_RUN_NAME = "rtdetr_cxr_test"

TARGET_CLASSES = [
    "Aortic enlargement",
    "Cardiomegaly",
    "Pleural effusion",
    "Pulmonary fibrosis",
    "Nodule/Mass",
]

for d in [YOLO_DATASET_DIR, YOLO_RUN_DIR, RTDETR_RUN_DIR, FASTER_RCNN_SAVE_DIR, PRETRAINED_DIR]:
    os.makedirs(d, exist_ok=True)

# ----- Hardware auto-detection -----
logical_cores = os.cpu_count() or 4
physical_cores = max(1, logical_cores // 2)
NUM_WORKERS = min(physical_cores, 8)
torch.set_num_threads(physical_cores)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    TRAIN_DEVICE = 0
else:
    TRAIN_DEVICE = "cpu"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Data source       : {DATA_ROOT}")
print(f"YOLO dataset      : {YOLO_DATASET_DIR}")
print(f"Pretrained dir    : {PRETRAINED_DIR}")
print(f"Classes           : {len(TARGET_CLASSES)}")
print(f"CPU cores         : ~{physical_cores} physical / {logical_cores} logical")
print(f"DataLoader workers: {NUM_WORKERS}")
print(f"Training device   : {TRAIN_DEVICE}")


# ----- Helpers -----
def ensure_pretrained(filename, model_class):
    """Ensure a pretrained weight file lives at PRETRAINED_DIR/filename."""
    target = Path(PRETRAINED_DIR) / filename
    if target.exists():
        return str(target)
    cwd_file = Path(filename)
    if not cwd_file.exists():
        print(f"Downloading {filename}...")
        model_class(filename)
    if cwd_file.exists():
        shutil.move(str(cwd_file), str(target))
        print(f"Moved {filename} -> {target}")
    return str(target)


def is_resumable(ckpt_path):
    """Check if an Ultralytics checkpoint has full resume state (epoch + optimizer).

    Ultralytics silently falls back to fresh training with defaults if you call
    .train(resume=True) on a checkpoint missing these fields. We guard against
    this by inspecting the checkpoint first.
    """
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if not isinstance(ckpt, dict):
            return False
        return ckpt.get("epoch", -1) >= 0 and ckpt.get("optimizer") is not None
    except Exception as e:
        print(f"  Cannot read checkpoint {ckpt_path}: {e}")
        return False


Data source       : Data/vindr_data
YOLO dataset      : Data/yolo_dataset
Pretrained dir    : /home/nguyendung/Documents/File_hoc_tap/Học_kỳ_10/Học thống kê/Đồ án nhóm/Code/deploy/Data/pretrained
Classes           : 5
CPU cores         : ~4 physical / 8 logical
DataLoader workers: 4
Training device   : cpu


## Data Preparation

In [5]:
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"train.csv not found at: {CSV_PATH}")
if not os.path.exists(META_PATH):
    raise FileNotFoundError(f"train_meta.csv not found at: {META_PATH}")

train_imgs_dir = os.path.join(YOLO_DATASET_DIR, "images", "train")
if os.path.exists(train_imgs_dir) and len(os.listdir(train_imgs_dir)) > 100:
    print(f"YOLO dataset already exists at {YOLO_DATASET_DIR}, skipping rebuild.")
    print("Delete that folder to force a rebuild.")
else:
    df = pd.read_csv(CSV_PATH)
    meta_df = pd.read_csv(META_PATH).rename(columns={"dim0": "height", "dim1": "width"})
    df = pd.merge(df, meta_df[["image_id", "height", "width"]], on="image_id", how="left")

    missing_meta = df["width"].isna().sum()
    if missing_meta > 0:
        print(f"Warning: dropping {missing_meta} annotations missing metadata.")
    df = df.dropna(subset=["width", "height"]).copy()

    class_to_id = {cls_name: idx for idx, cls_name in enumerate(TARGET_CLASSES)}
    df_filtered = df[df["class_name"].isin(TARGET_CLASSES)].copy()
    df_filtered["class_id"] = df_filtered["class_name"].map(class_to_id).astype(int)

    invalid_mask = (
        (df_filtered["x_max"] <= df_filtered["x_min"])
        | (df_filtered["y_max"] <= df_filtered["y_min"])
    )
    n_invalid = invalid_mask.sum()
    if n_invalid > 0:
        print(f"Warning: dropping {n_invalid} degenerate bboxes.")
    df_filtered = df_filtered[~invalid_mask].copy()

    df_filtered["x_center"] = ((df_filtered["x_min"] + df_filtered["x_max"]) / 2) / df_filtered["width"]
    df_filtered["y_center"] = ((df_filtered["y_min"] + df_filtered["y_max"]) / 2) / df_filtered["height"]
    df_filtered["w"] = (df_filtered["x_max"] - df_filtered["x_min"]) / df_filtered["width"]
    df_filtered["h"] = (df_filtered["y_max"] - df_filtered["y_min"]) / df_filtered["height"]

    unique_images = df_filtered["image_id"].unique()
    train_imgs, val_imgs = train_test_split(unique_images, test_size=0.2, random_state=42)
    print(f"Total images: {len(unique_images)} | Train: {len(train_imgs)} | Val: {len(val_imgs)}")


    def build_dataset(image_ids, subset_name):
        img_dir = os.path.join(YOLO_DATASET_DIR, "images", subset_name)
        label_dir = os.path.join(YOLO_DATASET_DIR, "labels", subset_name)
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(label_dir, exist_ok=True)

        subset_df = df_filtered[df_filtered["image_id"].isin(image_ids)]
        grouped = subset_df.groupby("image_id")
        missing = 0

        for img_id, group in tqdm(grouped, desc=f"Building {subset_name}"):
            src_img = os.path.join(IMG_SOURCE_DIR, f"{img_id}.png")
            if not os.path.exists(src_img):
                missing += 1
                continue
            dst_img = os.path.join(img_dir, f"{img_id}.png")
            if not os.path.exists(dst_img):
                shutil.copy(src_img, dst_img)
            label_file = os.path.join(label_dir, f"{img_id}.txt")
            labels = group[["class_id", "x_center", "y_center", "w", "h"]].values
            np.savetxt(label_file, labels, fmt="%d %.6f %.6f %.6f %.6f")

        if missing > 0:
            print(f"  Warning: {missing} source images not found.")

    build_dataset(train_imgs, "train")
    build_dataset(val_imgs, "val")
    print(f"Dataset ready at: {YOLO_DATASET_DIR}")


YOLO dataset already exists at Data/yolo_dataset, skipping rebuild.
Delete that folder to force a rebuild.


In [6]:
data_yaml = {
    "train": os.path.abspath(os.path.join(YOLO_DATASET_DIR, "images", "train")),
    "val": os.path.abspath(os.path.join(YOLO_DATASET_DIR, "images", "val")),
    "nc": len(TARGET_CLASSES),
    "names": list(TARGET_CLASSES),
}

with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml written: {YAML_PATH}")


data.yaml written: Data/yolo_dataset/data.yaml


## 1. YOLOv8 (10 epochs)

In [6]:
yolo_target_dir = os.path.join(YOLO_RUN_DIR, YOLO_RUN_NAME)
yolo_last_pt = os.path.join(yolo_target_dir, "weights", "last.pt")
yolo_best_pt = os.path.join(yolo_target_dir, "weights", "best.pt")

if os.path.exists(yolo_last_pt) and is_resumable(yolo_last_pt):
    # last.pt has epoch + optimizer state -> safe to resume
    print(f"Resuming YOLOv8 from: {yolo_last_pt}")
    yolo_model = YOLO(yolo_last_pt)
    yolo_results = yolo_model.train(resume=True)
elif os.path.exists(yolo_best_pt):
    # Training completed (or last.pt was stripped) - don't retrain
    print(f"Best weights already exist at {yolo_best_pt}.")
    print("Training appears complete. To retrain, delete the run folder:")
    print(f"  rm -rf {yolo_target_dir}")
else:
    # Fresh training
    print("Starting YOLOv8 training from scratch...")
    yolo_pretrained = ensure_pretrained("yolov8n.pt", YOLO)
    yolo_model = YOLO(yolo_pretrained)
    yolo_results = yolo_model.train(
        data=os.path.abspath(YAML_PATH),
        epochs=10,
        imgsz=512,
        batch=16,
        project=YOLO_RUN_DIR,
        name=YOLO_RUN_NAME,
        exist_ok=True,
        save_period=10,
        patience=0,
        workers=NUM_WORKERS,
        device=TRAIN_DEVICE,
    )

print(f"YOLOv8 done. Best weights: {yolo_best_pt}")


Best weights already exist at /home/nguyendung/Documents/File_hoc_tap/Học_kỳ_10/Học thống kê/Đồ án nhóm/Code/deploy/Data/YOLO_Runs/yolov8n_cxr_test/weights/best.pt.
Training appears complete. To retrain, delete the run folder:
  rm -rf /home/nguyendung/Documents/File_hoc_tap/Học_kỳ_10/Học thống kê/Đồ án nhóm/Code/deploy/Data/YOLO_Runs/yolov8n_cxr_test
YOLOv8 done. Best weights: /home/nguyendung/Documents/File_hoc_tap/Học_kỳ_10/Học thống kê/Đồ án nhóm/Code/deploy/Data/YOLO_Runs/yolov8n_cxr_test/weights/best.pt


## 2. RT-DETR (10 epochs)

In [ ]:
rtdetr_target_dir = os.path.join(RTDETR_RUN_DIR, RTDETR_RUN_NAME)
rtdetr_last_pt = os.path.join(rtdetr_target_dir, "weights", "last.pt")
rtdetr_best_pt = os.path.join(rtdetr_target_dir, "weights", "best.pt")

if os.path.exists(rtdetr_last_pt) and is_resumable(rtdetr_last_pt):
    print(f"Resuming RT-DETR from: {rtdetr_last_pt}")
    rtdetr_model = RTDETR(rtdetr_last_pt)
    rtdetr_results = rtdetr_model.train(resume=True)
elif os.path.exists(rtdetr_best_pt):
    print(f"Best weights already exist at {rtdetr_best_pt}.")
    print("Training appears complete. To retrain, delete the run folder:")
    print(f"  rm -rf {rtdetr_target_dir}")
else:
    print("Starting RT-DETR training from scratch...")
    rtdetr_pretrained = ensure_pretrained("rtdetr-l.pt", RTDETR)
    rtdetr_model = RTDETR(rtdetr_pretrained)
    rtdetr_results = rtdetr_model.train(
        data=os.path.abspath(YAML_PATH),
        epochs=10,
        imgsz=512,
        batch=8,
        project=RTDETR_RUN_DIR,
        name=RTDETR_RUN_NAME,
        exist_ok=True,
        save_period=10,
        patience=0,
        workers=NUM_WORKERS,
        device=TRAIN_DEVICE,
    )

print(f"RT-DETR done. Best weights: {rtdetr_best_pt}")


Starting RT-DETR training from scratch...
Ultralytics 8.4.52 🚀 Python-3.10.12 torch-2.8.0+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/nguyendung/Documents/File_hoc_tap/Học_kỳ_10/Học thống kê/Đồ án nhóm/Code/deploy/Data/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/nguyendung/Documents

## 3. Faster R-CNN (20 epochs)

In [7]:
class CXRDataset(Dataset):
    """Faster R-CNN dataset that reads YOLO labels and converts to Pascal VOC format."""

    def __init__(self, img_dir, label_dir, img_size=512):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.img_size = img_size
        self.imgs = sorted(
            [f for f in os.listdir(img_dir) if f.lower().endswith(".png")]
        )

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        img_resized = cv2.resize(
            img_rgb,
            (self.img_size, self.img_size),
            interpolation=cv2.INTER_LINEAR,
        )

        img = img_resized.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1).contiguous()

        label_path = os.path.join(
            self.label_dir,
            os.path.splitext(img_name)[0] + ".txt",
        )
        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    class_id, cx, cy, w, h = map(float, parts)

                    x_min = (cx - w / 2) * self.img_size
                    y_min = (cy - h / 2) * self.img_size
                    x_max = (cx + w / 2) * self.img_size
                    y_max = (cy + h / 2) * self.img_size

                    x_min = max(0.0, min(x_min, self.img_size - 1.0))
                    y_min = max(0.0, min(y_min, self.img_size - 1.0))
                    x_max = max(0.0, min(x_max, self.img_size - 1.0))
                    y_max = max(0.0, min(y_max, self.img_size - 1.0))

                    if x_max <= x_min or y_max <= y_min:
                        continue

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(class_id) + 1)  # +1: class 0 = background

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


def create_faster_rcnn_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


train_dataset = CXRDataset(
    os.path.join(YOLO_DATASET_DIR, "images", "train"),
    os.path.join(YOLO_DATASET_DIR, "labels", "train"),
)
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS,
)

val_dataset = CXRDataset(
    os.path.join(YOLO_DATASET_DIR, "images", "val"),
    os.path.join(YOLO_DATASET_DIR, "labels", "val"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS,
)

faster_rcnn = create_faster_rcnn_model(num_classes=len(TARGET_CLASSES) + 1).to(device)

print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")
print("Faster R-CNN ready.")


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /home/nguyendung/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:10<00:00, 15.6MB/s] 


Train images: 2535 | Val images: 608
Faster R-CNN ready.


In [ ]:
NUM_EPOCHS = 20
CHECKPOINT_EVERY = 10
GRAD_CLIP_NORM = 10.0
PATIENCE = 5  # Early stopping: stop if val loss does not improve for this many epochs

best_path = os.path.join(FASTER_RCNN_SAVE_DIR, "best_faster_rcnn.pth")
last_ckpt_path = os.path.join(FASTER_RCNN_SAVE_DIR, "last_faster_rcnn.pth")

params = [p for p in faster_rcnn.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
# Reduce LR by 10x every 3 epochs (aggressive regularization to prevent overfitting)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

start_epoch = 1
best_val_loss = float("inf")
patience_counter = 0

# ----- Resume logic -----
if os.path.exists(last_ckpt_path):
    print(f"Found Faster R-CNN checkpoint: {last_ckpt_path}")
    ckpt = torch.load(last_ckpt_path, map_location=device, weights_only=False)
    faster_rcnn.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    # Restore scheduler & patience state if available (older checkpoints may not have these)
    if "scheduler_state_dict" in ckpt:
        lr_scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    patience_counter = ckpt.get("patience_counter", 0)
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    if start_epoch > NUM_EPOCHS:
        print(f"Already trained {ckpt['epoch']}/{NUM_EPOCHS} epochs. Skipping.")
    else:
        print(f"Resuming from epoch {start_epoch} (best val: {best_val_loss:.4f}, patience: {patience_counter}/{PATIENCE})")

if start_epoch <= NUM_EPOCHS:
    print(f"Training Faster R-CNN epochs {start_epoch}-{NUM_EPOCHS}\n" + "=" * 50)

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        # --- Training phase ---
        faster_rcnn.train()
        train_loss_epoch = 0.0
        num_train_batches = 0

        train_loop = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Training")
        for i, (images, targets) in enumerate(train_loop):
            images = [img.to(device, non_blocking=True) for img in images]
            targets = [
                {k: v.to(device, non_blocking=True) for k, v in t.items()}
                for t in targets
            ]

            loss_dict = faster_rcnn(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if not torch.isfinite(losses):
                optimizer.zero_grad(set_to_none=True)
                del images, targets, loss_dict, losses
                continue

            optimizer.zero_grad(set_to_none=True)
            losses.backward()
            torch.nn.utils.clip_grad_norm_(faster_rcnn.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()

            train_loss_epoch += losses.item()
            num_train_batches += 1
            train_loop.set_postfix(loss=losses.item())
            del images, targets, loss_dict, losses

        avg_train_loss = train_loss_epoch / max(num_train_batches, 1)

        # --- Validation phase ---
        # torchvision FasterRCNN only returns loss_dict in .train() mode
        faster_rcnn.train()
        val_loss_epoch = 0.0
        num_val_batches = 0

        val_loop = tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Validation")
        with torch.no_grad():
            for images, targets in val_loop:
                images = [img.to(device, non_blocking=True) for img in images]
                targets = [
                    {k: v.to(device, non_blocking=True) for k, v in t.items()}
                    for t in targets
                ]
                loss_dict = faster_rcnn(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                if torch.isfinite(losses):
                    val_loss_epoch += losses.item()
                    num_val_batches += 1
                    val_loop.set_postfix(loss=losses.item())
                del images, targets, loss_dict, losses

        avg_val_loss = val_loss_epoch / max(num_val_batches, 1)

        print(
            f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
            f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        # Step LR scheduler
        lr_scheduler.step()

        # Early stopping + best checkpoint
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(faster_rcnn.state_dict(), best_path)
            print(f"  -> New best (val={best_val_loss:.4f}) saved.")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"  -> Val loss did not improve. Patience: {patience_counter}/{PATIENCE}")

        # Periodic checkpoint (state_dict only, for reference)
        if epoch % CHECKPOINT_EVERY == 0:
            ckpt_path = os.path.join(FASTER_RCNN_SAVE_DIR, f"epoch_{epoch}_faster_rcnn.pth")
            torch.save(faster_rcnn.state_dict(), ckpt_path)
            print(f"  -> Checkpoint at epoch {epoch}.")

        # Last full checkpoint (for resume)
        torch.save({
            "epoch": epoch,
            "model_state_dict": faster_rcnn.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": lr_scheduler.state_dict(),
            "best_val_loss": best_val_loss,
            "patience_counter": patience_counter,
        }, last_ckpt_path)

        # Early stopping trigger
        if patience_counter >= PATIENCE:
            print(f"  -> Early stopping triggered after {patience_counter} epochs without improvement.")
            break

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("=" * 50)
    print(f"Faster R-CNN training done.")
    print(f"  Best: {best_path}")
    print(f"  Last: {last_ckpt_path}")


Training Faster R-CNN epochs 1-20


Epoch 1/20 Training:   4%|▍         | 25/634 [12:10<5:14:33, 30.99s/it, loss=0.584]